**Silver: coin_markets**

In [0]:
# Clean & Standardize

from pyspark.sql.functions import col

df = spark.read.table("medallionarchitecture.bronze.coin_markets")

df_silver = df.select(
    col("id").alias("coin_id"),
    col("symbol"),
    col("name"),
    col("current_price").cast("double"),
    col("market_cap").cast("double"),
    col("total_volume").cast("double"),
    col("last_updated"),
    col("ingestion_time")
)

In [0]:
# DQ Rules 

df_silver = df_silver.filter(col("current_price") > 0)
df_silver = df_silver.dropDuplicates(["coin_id"])

In [0]:
# Save 

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallionarchitecture.silver.coin_markets")

**Silver : simple_price**

In [0]:
# clean

df = spark.read.table("medallionarchitecture.bronze.simple_price")

df_silver = df.select(
    col("coin"),
    col("price_usd").cast("double"),
    col("ingestion_time")
)

In [0]:
# save

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallionarchitecture.silver.simple_price")

**Silver: coin_details (SHOWCASE)** 
Only extract useful business fields

In [0]:
# Extract Important Fields and denormalize categories

from pyspark.sql.functions import col, explode

df = spark.read.table("medallionarchitecture.bronze.coin_details")

df_silver = df.select(
    col("id").alias("coin_id"),
    col("symbol"),
    col("name"),

    col("market_data.current_price.usd").alias("price_usd"),
    col("market_data.market_cap.usd").alias("market_cap_usd"),

    col("block_time_in_minutes"),

    explode(col("categories")).alias("category"),  # 🔥 flatten here

    col("ingestion_time")
)

In [0]:
# DQ Rule

df_silver = df_silver.filter(col("coin_id").isNotNull())
df_silver = df_silver.filter(col("category").isNotNull())
df_silver = df_silver.dropDuplicates(["coin_id", "category"])

In [0]:
# Save

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallionarchitecture.silver.coin_details")